In [11]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import ipywidgets as widgets
from IPython.display import display

In [3]:
queries = [
    ("What are the common symptoms of diabetes?", "symptoms"),
    ("How do I know if I have diabetes?", "symptoms"),
    ("List the warning signs of high blood sugar.", "symptoms"),
    ("Can frequent urination indicate diabetes?", "symptoms"),
    ("Tell me the main symptoms of type 2 diabetes.", "symptoms"),
    ("What indicates a diabetic emergency?", "symptoms"),
    ("What signs show a diabetic complication?", "symptoms"),
    ("What are the symptoms of hypoglycemia?", "symptoms"),
    ("What diet should a diabetic follow?", "diet"),
    ("What foods should I avoid with diabetes?", "diet"),
    ("Can I eat sweet fruits if I have diabetes?", "diet"),
    ("Is oatmeal good for managing blood sugar?", "diet"),
    ("Suggest a healthy meal plan for diabetes.", "diet"),
    ("Are carbohydrates bad for diabetic patients?", "diet"),
    ("Can I drink alcohol if I am diabetic?", "diet"),
    ("Suggest low glycemic index foods.", "diet"),
    ("What exercises are good for diabetic patients?", "exercise"),
    ("How often should a diabetic person work out?", "exercise"),
    ("Can regular walking help reduce blood sugar?", "exercise"),
    ("Is weight training safe for type 2 diabetes?", "exercise"),
    ("What is the best time for a diabetic to exercise?", "exercise"),
    ("Does physical activity lower insulin resistance?", "exercise"),
    ("Should I check blood sugar before working out?", "exercise"),
    ("What physical activities are recommended?", "exercise"),
    ("What medications are used to treat diabetes?", "medication"),
    ("Tell me about metformin usage.", "medication"),
    ("Are there side effects of diabetes drugs?", "medication"),
    ("How does insulin therapy work?", "medication"),
    ("When should I take my diabetes pills?", "medication"),
    ("Can medication cure diabetes completely?", "medication"),
    ("What are common injectable diabetes medications?", "medication"),
    ("How do SGLT2 inhibitors help with blood sugar?", "medication"),
    ("How should I monitor my blood glucose level?", "monitoring"),
    ("What is a normal fasting blood sugar reading?", "monitoring"),
    ("How often do I need to test HbA1c?", "monitoring"),
    ("What device is best for testing blood sugar?", "monitoring"),
    ("Why is my blood sugar high in the morning?", "monitoring"),
    ("Explain continuous glucose monitoring systems.", "monitoring"),
    ("What range is considered prediabetic?", "monitoring"),
    ("How do I track blood sugar patterns?", "monitoring")
]

df = pd.DataFrame(queries, columns=["query", "intent"])

taxonomy = {
    "Diabetes Care": {
        "Medical Management": ["Medication", "Monitoring"],
        "Lifestyle Interventions": ["Diet", "Exercise"]
    }
}

domain_concepts = [
    "Diabetes", "Insulin", "Glucose", "Metformin",
    "HbA1c", "Hypoglycemia", "Hyperglycemia", "Carbohydrates"
]

relationships = [
    ("Insulin", "regulates", "Glucose"),
    ("Metformin", "treats", "Diabetes"),
    ("HbA1c", "measures", "Glucose"),
    ("Hypoglycemia", "is", "Low Blood Sugar"),
    ("Hyperglycemia", "is", "High Blood Sugar"),
    ("Carbohydrates", "increase", "Glucose"),
    ("Exercise", "lowers", "Glucose"),
    ("Pancreas", "produces", "Insulin"),
    ("Diabetes", "requires", "Monitoring"),
    ("Diet", "manages", "Diabetes")
]

In [12]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    filtered_tokens = [lemmatizer.lemmatize(w) for w in tokens if w.isalnum() and w not in stop_words]
    return " ".join(filtered_tokens)

df['cleaned_query'] = df['query'].apply(preprocess)

In [5]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['cleaned_query'])
y = df['intent']

classifier = SVC(probability=True, kernel='linear', C=1.0)
classifier.fit(X, y)

KeyError: 'cleaned_query'

In [6]:
y_pred = classifier.predict(X)
accuracy = accuracy_score(y, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")

cm = confusion_matrix(y, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classifier.classes_, yticklabels=classifier.classes_, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

NameError: name 'classifier' is not defined

In [7]:
knowledge_base = {
    "symptoms": "Common diabetes symptoms include increased thirst, frequent urination, unexplained weight loss, fatigue, blurry vision, and slow-healing sores.",
    "diet": "A diabetic diet should focus on healthy carbohydrates (vegetables, fruits, whole grains), fiber-rich foods, fish, and good fats, while avoiding sugary drinks, trans fats, and highly processed items.",
    "exercise": "Physical activity helps lower blood sugar. Aim for at least 150 minutes of moderate-intensity aerobic exercise, like walking or cycling, along with strength training weekly.",
    "medication": "Common treatments include oral medications like Metformin to improve insulin sensitivity, or insulin therapy to directly control blood glucose levels.",
    "monitoring": "Blood glucose monitoring is critical. Fasting levels should ideally be between 80-130 mg/dL, and HbA1c checks should be performed at least twice a year."
}

clinical_terms = ["diabetes", "insulin", "glucose", "metformin", "sugar", "hba1c", "blood", "diet", "exercise", "symptom", "pill", "medication"]

In [8]:
def process_query(user_query):
    cleaned = preprocess(user_query)

    emergency_words = ["emergency", "unconscious", "coma", "severe", "chest pain", "breathless", "dying", "hospital"]
    if any(w in user_query.lower() for w in emergency_words):
        return "EMERGENCY ALERT: Please immediately contact emergency medical services (911) or visit the nearest emergency department. This system is for informational purposes only."

    has_clinical_term = any(term in cleaned for term in clinical_terms)
    if not has_clinical_term:
        return "UNSUPPORTED QUERY: This query falls outside the scope of the Diabetes Care Information Assistant. Please ask questions related to diabetes symptoms, diet, exercise, medication, or monitoring."

    vec = vectorizer.transform([cleaned])
    probs = classifier.predict_proba(vec)[0]
    max_prob = np.max(probs)
    pred_intent = classifier.classes_[np.argmax(probs)]

    if max_prob < 0.35:
        return "LOW CONFIDENCE: I am not completely sure what you are asking. Could you please rephrase your question? (Are you asking about diabetes symptoms, diet, medication, monitoring, or exercise?)"

    return f"[Intent Identified: {pred_intent.upper()} (Confidence: {max_prob:.2f})]\n\n{knowledge_base[pred_intent]}"

In [9]:
test_cases = [
    "What are the common symptoms of diabetes?",
    "Could you suggest some physical exercises that are healthy for someone with high blood sugar?",
    "What is the weather today in New York?",
    "Can you give me medication tips?",
    "Help me, my family member is unconscious and suffering from extreme breathing difficulty!"
]

for tc in test_cases:
    print(f"User Query: {tc}")
    print(f"Assistant: {process_query(tc)}")
    print("-" * 80)

User Query: What are the common symptoms of diabetes?


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [10]:
input_text = widgets.Text(value='', placeholder='Type your healthcare query...', description='Query:', layout=widgets.Layout(width='60%'))
output_html = widgets.HTML(value='')
submit_button = widgets.Button(description='Ask Assistant', button_style='primary')

def on_submit(b):
    query = input_text.value.strip()
    if query:
        response = process_query(query)
        formatted_response = response.replace("\n", "<br>")
        output_html.value = f"<div style='border:1px solid #ddd; padding:10px; border-radius:5px; margin-top:10px;'><b>Query:</b> {query}<br><br><b>Response:</b> {formatted_response}</div>"
    else:
        output_html.value = "Please type a valid query."

submit_button.on_click(on_submit)
display(widgets.VBox([widgets.HBox([input_text, submit_button]), output_html]))

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************
